In [1]:
import h5py
import torch
import numpy as np
from safetensors.torch import load_file
from tqdm.notebook import tqdm
import pandas as pd

class FeatureFinder:
    def __init__(self, activations_path, model_path, device="cuda:2"):
        self.device = torch.device(device)
        self.load_data(activations_path, model_path)
        
    def load_data(self, activations_path, model_path):
        """Load activations, texts and SAE model"""
        print("Loading data...")
        with h5py.File(activations_path, 'r') as f:
            self.activations = f['activations']
            self.activations = self.activations[:len(self.activations)//4]
            self.texts = f['text_refs'][:]
        
        self.sae_state = load_file(model_path)
        
        # Move model params to device
        self.encoder_w = self.sae_state['_orig_mod.encode.weight'].to(self.device).to(torch.bfloat16)
        self.encoder_b = self.sae_state['_orig_mod.encode.bias'].to(self.device).to(torch.bfloat16)
        print(self.encoder_b.dtype)
        
    def get_feature_activations(self, batch_size=4096):
        """Get feature activations in batches, keeping in bfloat16 on GPU"""
        print("Computing feature activations...")
        n_samples = len(self.activations)
        n_features = self.encoder_w.shape[0]
        
        # Initialize on GPU in bfloat16
        all_features = torch.zeros(
            (n_samples, n_features), 
            dtype=torch.bfloat16,
            device=self.device
        )
        
        for i in tqdm(range(0, n_samples, batch_size)):
            batch_end = min(i + batch_size, n_samples)
            # Convert batch to bfloat16
            batch = torch.tensor(
                self.activations[i:batch_end], 
                dtype=torch.bfloat16,
                device=self.device
            )
            
            # Forward pass through encoder
            with torch.no_grad():
                features = torch.relu(self.encoder_w @ batch.T + self.encoder_b.unsqueeze(1))
                all_features[i:batch_end] = features.T

        return all_features
    
    def analyze_feature_intervals(self, feature_activations, n_intervals=13, samples_per_interval=10):
        """Analyze features across activation intervals"""
        print("Analyzing feature intervals...")
        n_features = feature_activations.shape[1]
        feature_data = []
        
        for feat_idx in tqdm(range(n_features)):
            feat_activations = feature_activations[:, feat_idx]
            
            # Skip if feature never activates
            if feat_activations.max() <= 0:
                continue
                
            # Create intervals
            max_act = feat_activations.max()
            interval_bounds = np.linspace(0, max_act, n_intervals+1)
            
            # Analyze each interval
            for i in range(n_intervals):
                lower = interval_bounds[i]
                upper = interval_bounds[i+1]
                
                # Find examples in this interval
                mask = (feat_activations >= lower) & (feat_activations < upper)
                interval_indices = np.where(mask)[0]
                
                if len(interval_indices) == 0:
                    continue
                    
                # Sample examples from interval
                if len(interval_indices) > samples_per_interval:
                    interval_indices = np.random.choice(
                        interval_indices, 
                        samples_per_interval, 
                        replace=False
                    )
                
                # Record examples
                for idx in interval_indices:
                    feature_data.append({
                        'feature_id': feat_idx,
                        'interval': i,
                        'activation': feat_activations[idx],
                        'text': self.texts[idx],
                    })
                    
        return pd.DataFrame(feature_data)
    
    def get_feature_statistics(self, feature_activations):
        """Compute basic statistics for each feature"""
        stats = {
            'density': (feature_activations > 0).mean(axis=0),
            'mean_activation': feature_activations.mean(axis=0),
            'max_activation': feature_activations.max(axis=0),
            'activation_frequency': (feature_activations > 0).sum(axis=0)
        }
        return pd.DataFrame(stats)
    
    def find_interesting_features(self, feature_activations, min_density=1e-4, max_density=0.1):
        """Find potentially interesting features based on activation patterns"""
        stats = self.get_feature_statistics(feature_activations)
        
        # Filter features within density range
        interesting_features = stats[
            (stats['density'] >= min_density) & 
            (stats['density'] <= max_density)
        ].sort_values('max_activation', ascending=False)
        
        return interesting_features
    
    def analyze_features(self, batch_size=1024):
        """Run full feature analysis"""
        # Get feature activations
        feature_activations = self.get_feature_activations(batch_size)
        
        # Find interesting features
        interesting_features = self.find_interesting_features(feature_activations)
        print(f"\nFound {len(interesting_features)} potentially interesting features")
        
        # Get interval analysis for interesting features
        feature_examples = self.analyze_feature_intervals(feature_activations)
        
        return {
            'activations': feature_activations,
            'interesting_features': interesting_features,
            'feature_examples': feature_examples
        }

def display_feature_examples(feature_examples, feature_id):
    """Display examples for a specific feature across activation intervals"""
    feature_data = feature_examples[feature_examples['feature_id'] == feature_id]
    
    print(f"Examples for Feature {feature_id}:")
    print("-" * 80)
    
    for interval in sorted(feature_data['interval'].unique()):
        interval_examples = feature_data[feature_data['interval'] == interval]
        
        print(f"\nInterval {interval} (activation range):")
        for _, row in interval_examples.iterrows():
            print(f"Activation: {row['activation']:.4f}")
            print(f"Text: {row['text']}")
            print("-" * 40)
            
# Usage example:
finder = FeatureFinder(
    activations_path='/home/henry/Documents/PythonProjects/open-concept-steering/data/residual_stream_activations_llama1b_bf16.h5',
    model_path='/home/henry/Documents/PythonProjects/open-concept-steering/out/sae_16k/final_model.safetensors',
    device="cuda:2"
)

# Run the analysis
results = finder.analyze_features()

# Look at the first few interesting features
print(results['interesting_features'].head())

# Look at examples for one of the interesting features
feature_id = results['interesting_features'].index[0]
display_feature_examples(results['feature_examples'], feature_id)

Loading data...
torch.bfloat16
Computing feature activations...


OutOfMemoryError: CUDA out of memory. Tried to allocate 762.95 GiB. GPU 2 has a total capacity of 23.60 GiB of which 23.22 GiB is free. Including non-PyTorch memory, this process has 362.00 MiB memory in use. Of the allocated memory 64.03 MiB is allocated by PyTorch, and 1.97 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)